<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/04_generative_ai/genai_projects/experiment_basic_rag_langchain_chromadb_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Basic Retrieval Augmented Generation (RAG) System

First, let's install the required libraries. We'll use `langchain` for orchestrating the RAG pipeline, `chromadb` as our vector store, `sentence-transformers` for creating embeddings, and `google-generativeai` for the Large Language Model (LLM).

In [1]:
!pip install -U langchain-google-genai langchain chromadb pypdf sentence-transformers langchain_community langchain_text_splitters
!pip install requests==2.32.4

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.43.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry

To use the Gemini API for generation, you'll need an API key. If you don't already have one, create a key in Google AI Studio. In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then, we'll load it securely.

In [2]:
from google.colab import userdata
import os

# Set the GOOGLE_API_KEY environment variable
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

Now, let's prepare some sample data. I will create a dummy text document to serve as our knowledge base. In a real application, you would load your documents from files (e.g., PDFs, text files) or other data sources.

In [3]:
# Create a dummy text file for demonstration
with open('sample_document.txt', 'w') as f:
    f.write("The quick brown fox jumps over the lazy dog. This is a test document for RAG. "
            "RAG stands for Retrieval Augmented Generation. It combines information retrieval "
            "with text generation to provide more accurate and up-to-date responses. "
            "The capital of France is Paris. The Eiffel Tower is in Paris. "
            "The sun is a star. Planets revolve around the sun.")

# Load the document
from langchain_community.document_loaders import TextLoader

loader = TextLoader("sample_document.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")
print(documents[0].page_content[:200]) # Display first 200 characters of the first document

/tmp/ipykernel_11290/4129743358.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 1 document(s).
The quick brown fox jumps over the lazy dog. This is a test document for RAG. RAG stands for Retrieval Augmented Generation. It combines information retrieval with text generation to provide more accu


Next, we'll split the loaded document into smaller, manageable chunks. This is crucial for efficient retrieval, as embedding large documents directly can be less effective and computationally expensive. We'll use `RecursiveCharacterTextSplitter` for this.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,  # Each chunk will have a maximum of 100 characters
    chunk_overlap=20, # 20 characters will overlap between consecutive chunks
    length_function=len,
    add_start_index=True,
)
splits = text_splitter.split_documents(documents)

print(f"Split into {len(splits)} chunks.")
for i, split in enumerate(splits[:3]):
    print(f"Chunk {i+1}: {split.page_content}")

Split into 5 chunks.
Chunk 1: The quick brown fox jumps over the lazy dog. This is a test document for RAG. RAG stands for
Chunk 2: RAG. RAG stands for Retrieval Augmented Generation. It combines information retrieval with text
Chunk 3: retrieval with text generation to provide more accurate and up-to-date responses. The capital of


Now, we'll create embeddings for our document chunks. Embeddings convert text into numerical vectors that capture their semantic meaning. We'll use the `HuggingFaceEmbeddings` for this, and then store these embeddings in a `Chroma` vector database. Chroma is a lightweight and easy-to-use vector store perfect for this kind of application.

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embeddings model
# We'll use a common Sentence Transformer model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a Chroma vector store from the document splits and embeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print(f"Vectorstore created with {vectorstore._collection.count()} entries.")

# Let's test the retriever with a sample query
retriever = vectorstore.as_retriever()

query = "What is the capital of France?"
retrieved_docs = retriever.invoke(query)

print(f"\nRetrieved documents for query '{query}':")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}: {doc.page_content}")

/tmp/ipykernel_11290/1835849664.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 5 entries.

Retrieved documents for query 'What is the capital of France?':
Document 1: The capital of France is Paris. The Eiffel Tower is in Paris. The sun is a star. Planets revolve
Document 2: retrieval with text generation to provide more accurate and up-to-date responses. The capital of
Document 3: Planets revolve around the sun.
Document 4: The quick brown fox jumps over the lazy dog. This is a test document for RAG. RAG stands for


With our vector store and retriever set up, the next step is to integrate a Large Language Model (LLM) to generate responses based on the retrieved information. We'll use the Gemini Pro model from Google for this.

In [12]:
import langchain
print(langchain.__version__)

1.3.11


In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import RetrievalQA

# Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# Create a RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # 'stuff' combines all retrieved documents into one prompt
    retriever=retriever,
    return_source_documents=True
)

# Example query
question = "Tell me about RAG."
result = qa_chain.invoke({"query": question}) # type: ignore

print(f"\nQuery: {question}")
print(f"Response: {result['result']}")
print("\nSource Documents:")
for doc in result['source_documents']:
    print(f"- {doc.page_content}")


Query: Tell me about RAG.
Response: RAG stands for Retrieval Augmented Generation. It combines information retrieval with text generation to provide more accurate and up-to-date responses.

Source Documents:
- RAG. RAG stands for Retrieval Augmented Generation. It combines information retrieval with text
- The quick brown fox jumps over the lazy dog. This is a test document for RAG. RAG stands for
- retrieval with text generation to provide more accurate and up-to-date responses. The capital of
- Planets revolve around the sun.


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import RetrievalQA

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

result = qa_chain.invoke({"query": "Tell me about RAG."}) # type: ignore

print(result)

{'query': 'Tell me about RAG.', 'result': 'RAG stands for Retrieval Augmented Generation. It combines information retrieval with text generation to provide more accurate and up-to-date responses.', 'source_documents': [Document(metadata={'source': 'sample_document.txt', 'start_index': 73}, page_content='RAG. RAG stands for Retrieval Augmented Generation. It combines information retrieval with text'), Document(metadata={'source': 'sample_document.txt', 'start_index': 0}, page_content='The quick brown fox jumps over the lazy dog. This is a test document for RAG. RAG stands for'), Document(metadata={'source': 'sample_document.txt', 'start_index': 149}, page_content='retrieval with text generation to provide more accurate and up-to-date responses. The capital of'), Document(metadata={'source': 'sample_document.txt', 'start_index': 312}, page_content='Planets revolve around the sun.')]}


This completes our basic RAG system! You can now ask questions, and the system will retrieve relevant information from your document and then generate a coherent answer using the LLM.

Now, we'll create embeddings for our document chunks. Embeddings convert text into numerical vectors that capture their semantic meaning. We'll use the `HuggingFaceEmbeddings` for this, and then store these embeddings in a `Chroma` vector database. Chroma is a lightweight and easy-to-use vector store perfect for this kind of application.

In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embeddings model
# We'll use a common Sentence Transformer model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a Chroma vector store from the document splits and embeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print(f"Vectorstore created with {vectorstore._collection.count()} entries.")

# Let's test the retriever with a sample query
retriever = vectorstore.as_retriever()

query = "What is the capital of France?"
retrieved_docs = retriever.invoke(query)

print(f"\nRetrieved documents for query '{query}':")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}: {doc.page_content}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore created with 10 entries.

Retrieved documents for query 'What is the capital of France?':
Document 1: The capital of France is Paris. The Eiffel Tower is in Paris. The sun is a star. Planets revolve
Document 2: The capital of France is Paris. The Eiffel Tower is in Paris. The sun is a star. Planets revolve
Document 3: retrieval with text generation to provide more accurate and up-to-date responses. The capital of
Document 4: retrieval with text generation to provide more accurate and up-to-date responses. The capital of


With our vector store and retriever set up, the next step is to integrate a Large Language Model (LLM) to generate responses based on the retrieved information. We'll use the Gemini Pro model from Google for this.

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import RetrievalQA

# Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# Create a RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # 'stuff' combines all retrieved documents into one prompt
    retriever=retriever,
    return_source_documents=True
)

# Example query
question = "Tell me about RAG."
result = qa_chain.invoke({"query": question})

print(f"\nQuery: {question}")
print(f"Response: {result['result']}")
print("\nSource Documents:")
for doc in result['source_documents']:
    print(f"- {doc.page_content}")


Query: Tell me about RAG.
Response: RAG stands for Retrieval Augmented Generation. It combines information retrieval with text.

Source Documents:
- RAG. RAG stands for Retrieval Augmented Generation. It combines information retrieval with text
- RAG. RAG stands for Retrieval Augmented Generation. It combines information retrieval with text
- The quick brown fox jumps over the lazy dog. This is a test document for RAG. RAG stands for
- The quick brown fox jumps over the lazy dog. This is a test document for RAG. RAG stands for


This completes our basic RAG system! You can now ask questions, and the system will retrieve relevant information from your document and then generate a coherent answer using the LLM.